In [8]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from statsmodels.formula.api import ols

In [9]:
# Data: Test scores for each group

data = {
    'scores': [
        85, 78, 91, 87, 90,   # Method 1 - Lecture
        88, 84, 92, 89, 85,   # Method 1 - Lab
        75, 80, 78, 82, 77,   # Method 2 - Lecture
        83, 88, 81, 86, 85    # Method 2 - Lab
    ],

    'method': ['Method 1'] * 10 + ['Method 2'] * 10,

    'class_type': ['Lecture'] * 5 + ['Lab'] * 5 +
                  ['Lecture'] * 5 + ['Lab'] * 5
}

# Create a DataFrame
df = pd.DataFrame(data)

df

,scores,method,class_type
0,85,Method 1,Lecture
1,78,Method 1,Lecture
2,91,Method 1,Lecture
3,87,Method 1,Lecture
4,90,Method 1,Lecture
5,88,Method 1,Lab
6,84,Method 1,Lab
7,92,Method 1,Lab
8,89,Method 1,Lab
9,85,Method 1,Lab


In [10]:
# Perform Two-Way ANOVA

model = ols(
    'scores ~ C(method) + C(class_type) + C(method):C(class_type)',
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

# Display the ANOVA table
print(anova_table)

                         sum_sq    df          F    PR(>F)
C(method)                 145.8   1.0  11.302326  0.003967
C(class_type)              72.2   1.0   5.596899  0.030953
C(method):C(class_type)    28.8   1.0   2.232558  0.154589
Residual                  206.4  16.0        NaN       NaN


In [11]:
# Extracting necessary values for interpretation

ssa = anova_table['sum_sq']['C(method)']
ssb = anova_table['sum_sq']['C(class_type)']
ssab = anova_table['sum_sq']['C(method):C(class_type)']
sse = anova_table['sum_sq']['Residual']

sst = ssa + ssb + ssab + sse


df_a = anova_table['df']['C(method)']
df_b = anova_table['df']['C(class_type)']
df_ab = anova_table['df']['C(method):C(class_type)']
df_e = anova_table['df']['Residual']

In [12]:
# Mean Squares

msa = ssa / df_a
msb = ssb / df_b
msab = ssab / df_ab
mse = sse / df_e


# F-statistics

f_a = msa / mse
f_b = msb / mse
f_ab = msab / mse


# Display F-statistics

print(f"F-statistic for Method: {f_a:.4f}")
print(f"F-statistic for Class Type: {f_b:.4f}")
print(f"F-statistic for Interaction: {f_ab:.4f}")

F-statistic for Method: 11.3023
F-statistic for Class Type: 5.5969
F-statistic for Interaction: 2.2326


In [13]:
# Critical values and decision

alpha = 0.05

from scipy.stats import f as f_dist


# Critical value for F-distribution

f_critical = f_dist.ppf(
    1 - alpha,
    df_a,
    df_e
)

print(f"\nCritical F-value at alpha={alpha}: {f_critical:.4f}")


Critical F-value at alpha=0.05: 4.4940


In [14]:
# Final Decision

decisions = {
    "Method": f_a > f_critical,
    "Class Type": f_b > f_critical,
    "Interaction": f_ab > f_critical
}

for key, value in decisions.items():

    if value:
        print(f"The null hypothesis for {key} is rejected (significant effect).")

    else:
        print(f"The null hypothesis for {key} is not rejected (no significant effect).")

The null hypothesis for Method is rejected (significant effect).
The null hypothesis for Class Type is rejected (significant effect).
The null hypothesis for Interaction is not rejected (no significant effect).
